# Homework 4 — Couverture quadratique par réseau de neurones (Call BS 1D)

On considère un call européen dans le modèle de Black-Scholes 1D.
Le but est de construire une stratégie de couverture discrète, en moyenne quadratique optimale, via des réseaux de neurones, puis de comparer les fonctions de hedge obtenues avec le **delta BS**.

## Cadre

Dates de couverture : \(t_k=kT/N\), \(k=0,\dots,N\).

On travaille avec les quantités actualisées :
\[
\widetilde S_t=e^{-rt}S_t,\qquad \widetilde H=e^{-rT}(S_T-K)_+.
\]

Objectif (couverture quadratique discrète) :
\[
\min_{V_0,(\phi_k)}\;\mathbb E\!\left[\left(\widetilde H-V_0-\sum_{k=0}^{N-1}\phi_k(S_{t_k})\,\Delta\widetilde S_{k+1}ight)^2ight].
\]

On apprend numériquement les fonctions \(\phi_k(\cdot)\) par régression non-linéaire (MLP).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import norm
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
def bs_call_price_delta(S, K, r, sigma, tau):
    S = np.asarray(S, dtype=float)
    eps = 1e-14

    if tau <= 0:
        price = np.maximum(S - K, 0.0)
        delta = (S > K).astype(float)
        return price, delta

    vol = sigma * np.sqrt(tau)
    d1 = (np.log(np.maximum(S, eps) / K) + (r + 0.5 * sigma**2) * tau) / vol
    d2 = d1 - vol

    price = S * norm.cdf(d1) - K * np.exp(-r * tau) * norm.cdf(d2)
    delta = norm.cdf(d1)
    return price, delta


def simulate_bs_paths(S0, r, sigma, T, N, n_paths, seed=123):
    dt = T / N
    rng = np.random.default_rng(seed)

    Z = rng.standard_normal((n_paths, N))
    S = np.empty((n_paths, N + 1), dtype=float)
    S[:, 0] = S0

    drift = (r - 0.5 * sigma**2) * dt
    vol = sigma * np.sqrt(dt)

    for k in range(N):
        S[:, k + 1] = S[:, k] * np.exp(drift + vol * Z[:, k])

    return S


def discounted_stock(S, r, T, N):
    dt = T / N
    times = np.arange(N + 1) * dt
    return S * np.exp(-r * times)[None, :]

In [ ]:
def fit_nn_1d(x, y, nn_cfg):
    X = np.asarray(x, dtype=float).reshape(-1, 1)
    y = np.asarray(y, dtype=float).ravel()

    mlp = MLPRegressor(
        hidden_layer_sizes=nn_cfg.get('hidden_layer_sizes', (32, 32)),
        activation='relu',
        solver='adam',
        learning_rate_init=nn_cfg.get('learning_rate_init', 1e-3),
        max_iter=nn_cfg.get('max_iter', 200),
        batch_size=nn_cfg.get('batch_size', 128),
        random_state=nn_cfg.get('random_state', 123),
        early_stopping=False,
    )

    normalize = nn_cfg.get('normalize', True)
    if normalize:
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('mlp', mlp),
        ])
    else:
        model = mlp

    model.fit(X, y)
    return model


def predict_nn_1d(model, x):
    X = np.asarray(x, dtype=float).reshape(-1, 1)
    return model.predict(X)

In [ ]:
def train_quadratic_hedge_nn(
    S0, K, r, sigma, T, N,
    n_paths=30000,
    seed=7,
    nn_cfg=None,
    n_inner=2,
):
    if nn_cfg is None:
        nn_cfg = {}

    S = simulate_bs_paths(S0, r, sigma, T, N, n_paths=n_paths, seed=seed)
    S_tilde = discounted_stock(S, r, T, N)

    H_tilde = np.exp(-r * T) * np.maximum(S[:, -1] - K, 0.0)

    V_next = H_tilde.copy()

    models = [None] * N
    hedge_samples = np.zeros((n_paths, N), dtype=float)

    for k in range(N - 1, -1, -1):
        s_k = S[:, k]
        dS_tilde = S_tilde[:, k + 1] - S_tilde[:, k]

        # Initialisation h_k = 0 puis alternance g/h
        h_pred = np.zeros_like(s_k)

        for _ in range(n_inner):
            target_g = V_next - h_pred * dS_tilde
            g_model = fit_nn_1d(s_k, target_g, nn_cfg)
            g_pred = predict_nn_1d(g_model, s_k)

            num_target = dS_tilde * (V_next - g_pred)
            den_target = dS_tilde**2

            num_model = fit_nn_1d(s_k, num_target, nn_cfg)
            den_model = fit_nn_1d(s_k, den_target, nn_cfg)

            num_pred = predict_nn_1d(num_model, s_k)
            den_pred = np.maximum(predict_nn_1d(den_model, s_k), 1e-10)

            h_pred = num_pred / den_pred

        models[k] = {
            'g_model': g_model,
            'num_model': num_model,
            'den_model': den_model,
        }

        hedge_samples[:, k] = h_pred
        V_next = g_pred

    V0 = float(np.mean(V_next))

    return {
        'V0': V0,
        'models': models,
        'S': S,
        'S_tilde': S_tilde,
        'H_tilde': H_tilde,
        'hedge_samples': hedge_samples,
    }


def hedge_function_from_models(models, k, s_grid):
    num = predict_nn_1d(models[k]['num_model'], s_grid)
    den = np.maximum(predict_nn_1d(models[k]['den_model'], s_grid), 1e-10)
    return num / den

In [ ]:
# Paramètres (modifiables)
S0 = 100
K = 100
r = 0.02
sigma = 0.2
T = 1.0
N = 12

n_paths = 30000
seed = 7

nn_cfg = {
    'hidden_layer_sizes': (32, 32),
    'learning_rate_init': 1e-3,
    'max_iter': 200,
    'batch_size': 128,
    'normalize': True,
    'random_state': 7,
}

res = train_quadratic_hedge_nn(
    S0=S0, K=K, r=r, sigma=sigma, T=T, N=N,
    n_paths=n_paths,
    seed=seed,
    nn_cfg=nn_cfg,
    n_inner=2,
)

print(f"Capital initial estimé (actualisé) V0 : {res['V0']:.6f}")

bs_price_0, bs_delta_0 = bs_call_price_delta(S0, K, r, sigma, T)
print(f"Prix BS à t=0 (actualisé via formule BS standard non actualisée): {bs_price_0:.6f}")
print(f"Delta BS à t=0 : {bs_delta_0:.6f}")

In [ ]:
# Comparaison des fonctions hedge apprises avec le delta BS
s_grid = np.linspace(40, 180, 400)

# Dates à afficher
k_list = [0, 2, 4, 6, 8, 10]
k_list = [k for k in k_list if k < N]

fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True, sharey=True)
axes = axes.ravel()

for j, k in enumerate(k_list):
    tau = T - (k * T / N)

    hedge_nn = hedge_function_from_models(res['models'], k, s_grid)
    _, delta_bs = bs_call_price_delta(s_grid, K, r, sigma, tau)

    axes[j].plot(s_grid, hedge_nn, lw=2, label='Hedge NN')
    axes[j].plot(s_grid, delta_bs, lw=2, ls='--', label='Delta BS')
    axes[j].set_title(f'k={k}, t={k*T/N:.2f}')
    axes[j].grid(alpha=0.3)

axes[0].legend(fontsize=9)
fig.suptitle('Comparaison hedge quadratique NN vs delta BS', fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# Mesure simple de l'écart moyen (RMSE) entre hedge NN et delta BS
print('RMSE(hedge NN - delta BS) par date:')

for k in range(N):
    s_k = res['S'][:, k]
    tau = T - (k * T / N)

    hedge_k = hedge_function_from_models(res['models'], k, s_k)
    _, delta_k = bs_call_price_delta(s_k, K, r, sigma, tau)

    rmse = np.sqrt(np.mean((hedge_k - delta_k)**2))
    print(f'k={k:2d}, t={k*T/N:.2f}, RMSE={rmse:.5f}')

## Commentaire

Le hedge appris par réseau de neurones suit globalement la forme du delta BS (courbe croissante en fonction de \(S\)).
Les écarts viennent principalement de :
- la discrétisation temporelle de la couverture ;
- l'erreur Monte Carlo ;
- l'erreur d'approximation/optimisation du réseau.

Avec davantage de scénarios et un réglage plus fin des hyperparamètres, les deux fonctions se rapprochent en général.